# TIMMD — 00: Carga de Datos

Este notebook es el punto de entrada del proyecto. Se conecta a la API de StatsBomb, permite explorar competencias y partidos disponibles, y descarga los eventos y alineaciones de un partido específico. Los datos se guardan en Google Drive en formato Parquet para ser consumidos por el resto de los módulos sin necesidad de volver a llamar a la API.

## 1. Configuración del entorno

Instalación de dependencias, montaje de Google Drive y definición de la carpeta de datos compartida.
Si la carpeta `TIMMD` no existe en tu unidad, seguí el link para agregarla como acceso directo.

In [3]:
!pip install statsbombpy --quiet

import pandas as pd
from statsbombpy import sb

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Carpeta compartida del proyecto
# Si no tenés acceso: abrí este link y elegí 'Agregar acceso directo a Mi unidad' con el nombre TIMMD
# https://drive.google.com/drive/folders/1JCn2VRcg4vnjB5G0cq40gpGMgddG1PQR
DATA_DIR = "/content/drive/MyDrive/TIMMD/data"

if not os.path.exists(DATA_DIR):
    print("⚠️  No encontramos la carpeta de datos.")
    print("   Seguí las instrucciones del comentario de arriba y volvé a correr esta celda.")
else:
    print(f"✓ Carpeta de datos encontrada — {len(os.listdir(DATA_DIR))} archivos")

Mounted at /content/drive
✓ Carpeta de datos encontrada — 60 archivos


## 2. Exploración de competencias y partidos

Listado de todas las competencias disponibles en StatsBomb Open Data.
Seleccioná la competencia y temporada de interés para ver los partidos disponibles y elegir el `MATCH_ID`.

In [5]:
competencias = sb.competitions()
print(competencias[["competition_id", "competition_name", "season_id", "season_name"]].to_string())

/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


    competition_id         competition_name  season_id season_name
0                9            1. Bundesliga        281   2023/2024
1                9            1. Bundesliga         27   2015/2016
2             1267   African Cup of Nations        107        2023
3               16         Champions League          4   2018/2019
4               16         Champions League          1   2017/2018
5               16         Champions League          2   2016/2017
6               16         Champions League         27   2015/2016
7               16         Champions League         26   2014/2015
8               16         Champions League         25   2013/2014
9               16         Champions League         24   2012/2013
10              16         Champions League         23   2011/2012
11              16         Champions League         22   2010/2011
12              16         Champions League         21   2009/2010
13              16         Champions League         41   2008/

In [6]:
# Modificá competition_id y season_id según la competencia que querés explorar
partidos = sb.matches(competition_id=9, season_id=281)
print(partidos[["match_id", "home_team", "away_team", "home_score", "away_score", "match_date"]].to_string())

/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


    match_id                 home_team                 away_team  home_score  away_score  match_date
0    3895292              Union Berlin          Bayer Leverkusen           0           1  2024-04-06
1    3895320          Bayer Leverkusen             VfB Stuttgart           2           2  2024-04-27
2    3895158          Bayer Leverkusen         Borussia Dortmund           1           1  2023-12-03
3    3895107          Bayer Leverkusen                   FC Köln           3           0  2023-10-08
4    3895340                    Bochum          Bayer Leverkusen           0           5  2024-05-12
5    3895286          Bayer Leverkusen                Hoffenheim           2           1  2024-03-30
6    3895302          Bayer Leverkusen             Werder Bremen           5           0  2024-04-14
7    3895333       Eintracht Frankfurt          Bayer Leverkusen           1           5  2024-05-05
8    3895348          Bayer Leverkusen                  Augsburg           2           1  2

## 3. Carga del partido

Descarga de eventos y alineaciones del partido seleccionado mediante su `MATCH_ID`.
Este es el único valor que cambia entre corridas — todo lo demás se ajusta automáticamente.

In [7]:
MATCH_ID = 3895153                                 # ← único valor a cambiar

eventos = sb.events(match_id=MATCH_ID)

lineups    = sb.lineups(match_id=MATCH_ID)
lineup_df  = pd.concat(lineups.values(), ignore_index=True)
lineup_df["nombre_display"] = lineup_df["player_nickname"].where(
    lineup_df["player_nickname"].notna(), lineup_df["player_name"])

equipos      = eventos["team"].dropna().unique().tolist()
equipo_local = equipos[0]
equipo_visit = equipos[1]

print(f"✓ Partido {MATCH_ID} cargado — {eventos.shape[0]} eventos")
print(f"  {equipo_local} vs {equipo_visit}")

/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


✓ Partido 3895153 cargado — 4520 eventos
  Werder Bremen vs Bayer Leverkusen


/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


## 4. Guardado en Drive

Exporta los datos del partido a la carpeta compartida en formato Parquet (eventos y alineaciones) y JSON (metadata del partido).
Una vez guardados, los módulos 01 a 04 pueden correr sin necesidad de volver a llamar a la API.

In [8]:
eventos.to_parquet(f"{DATA_DIR}/eventos_{MATCH_ID}.parquet", index=False)
lineup_df.to_parquet(f"{DATA_DIR}/lineups_{MATCH_ID}.parquet", index=False)

pd.Series({
    "equipo_local": equipo_local,
    "equipo_visit": equipo_visit,
}).to_json(f"{DATA_DIR}/meta_{MATCH_ID}.json")

print(f"✓ Partido {MATCH_ID} guardado en {DATA_DIR}")
print(f"  {equipo_local} vs {equipo_visit}")

✓ Partido 3895153 guardado en /content/drive/MyDrive/TIMMD/data
  Werder Bremen vs Bayer Leverkusen
